In [0]:
from datetime import datetime
from decimal import Decimal

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

from delta.tables import DeltaTable


# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

silver_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipments/"
)


# ============================================================
# 2. ÉTAT DE LA TABLE AVANT MERGE
# ============================================================

silver_shipments_before = (
    spark.read
    .format("delta")
    .load(silver_shipments_path)
)

count_before = silver_shipments_before.count()

print(
    "Nombre de lignes AVANT MERGE :",
    count_before
)

print("\n=== S0131 AVANT MERGE ===")

display(
    silver_shipments_before
    .filter(
        F.col("shipment_id") == "S0131"
    )
    .select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "shipment_time",
        "expected_delivery_time",
        "actual_delivery_time",
        "distance_km",
        "shipment_status",
        "updated_at",
        "batch_id"
    )
)


# ============================================================
# 3. MINI-BATCH DE TEST
# ============================================================

merge_test_shipments = [

    # --------------------------------------------------------
    # CAS 1 : nouveau shipment
    # S0135 n'existe pas encore
    # => INSERT
    #
    # O0135 existe maintenant dans silver_orders.
    # --------------------------------------------------------
    (
        "S0135",
        "O0135",
        "C002",
        datetime(2026, 1, 12, 12, 0),
        datetime(2026, 1, 15, 12, 0),
        None,
        "Rabat",
        "Casablanca",
        Decimal("95.00"),
        "CREATED",
        datetime(2026, 1, 12, 12, 0)
    ),

    # --------------------------------------------------------
    # CAS 2 : mise à jour de S0131
    #
    # On conserve son shipment_time historique.
    # Seul son état métier évolue :
    # CREATED -> SHIPPED
    # --------------------------------------------------------
    (
        "S0131",
        "O0131",
        "C001",
        datetime(2026, 1, 10, 10, 0),
        datetime(2026, 1, 12, 10, 0),
        None,
        "Rabat",
        "Rabat",
        Decimal("25.00"),
        "SHIPPED",
        datetime(2026, 1, 12, 13, 0)
    )
]


# ============================================================
# 4. SCHÉMA
# ============================================================

merge_shipments_schema = StructType([
    StructField("shipment_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("carrier_id", StringType(), False),
    StructField("shipment_time", TimestampType(), False),
    StructField("expected_delivery_time", TimestampType(), False),
    StructField("actual_delivery_time", TimestampType(), True),
    StructField("origin_city", StringType(), False),
    StructField("destination_city", StringType(), False),
    StructField("distance_km", DecimalType(10, 2), False),
    StructField("shipment_status", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])


# ============================================================
# 5. DATAFRAME SOURCE
# ============================================================

merge_source = (
    spark.createDataFrame(
        merge_test_shipments,
        schema=merge_shipments_schema
    )

    .withColumn(
        "source_file",
        F.lit("phase4_merge_test")
    )

    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "batch_id",
        F.lit("merge_test_002")
    )
)

print("\n=== SOURCE DU MERGE ===")

display(
    merge_source
    .orderBy("shipment_id")
)


# ============================================================
# 6. TABLE DELTA CIBLE
# ============================================================

silver_shipments_delta = (
    DeltaTable.forPath(
        spark,
        silver_shipments_path
    )
)


# ============================================================
# 7. MERGE
# ============================================================

(
    silver_shipments_delta.alias("target")

    .merge(
        merge_source.alias("source"),
        "target.shipment_id = source.shipment_id"
    )

    # Mise à jour seulement si la source est plus récente
    .whenMatchedUpdateAll(
        condition="""
            source.updated_at > target.updated_at
        """
    )

    # Nouveau shipment
    .whenNotMatchedInsertAll()

    .execute()
)

print("\nMERGE shipments terminé.")


# ============================================================
# 8. ÉTAT APRÈS MERGE
# ============================================================

silver_shipments_after = (
    spark.read
    .format("delta")
    .load(silver_shipments_path)
)

count_after = silver_shipments_after.count()

print(
    "Nombre de lignes AVANT MERGE :",
    count_before
)

print(
    "Nombre de lignes APRÈS MERGE :",
    count_after
)

print(
    "Différence :",
    count_after - count_before
)


# ============================================================
# 9. VÉRIFIER L'INSERT : S0135
# ============================================================

print("\n=== S0135 : INSERT ===")

display(
    silver_shipments_after
    .filter(
        F.col("shipment_id") == "S0135"
    )
    .select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "shipment_time",
        "shipment_status",
        "updated_at",
        "batch_id"
    )
)


# ============================================================
# 10. VÉRIFIER L'UPDATE : S0131
# ============================================================

print("\n=== S0131 : UPDATE ===")

display(
    silver_shipments_after
    .filter(
        F.col("shipment_id") == "S0131"
    )
    .select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "shipment_time",
        "shipment_status",
        "updated_at",
        "batch_id"
    )
)